In [4]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [5]:
train_df = pd.read_csv("train.csv")

In [6]:
test_df = pd.read_csv("test.csv")

In [7]:
train_df.shape

(8693, 14)

In [8]:
test_df.shape

(4277, 13)

In [9]:
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [10]:
train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [11]:
train_df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [12]:
train_df['Transported'].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

In [13]:
train_df['CryoSleep'].value_counts(dropna=False)

CryoSleep
False    5439
True     3037
NaN       217
Name: count, dtype: int64

In [14]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

train_df[spend_cols].describe()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,224.687617,458.077203,173.729169,311.138778,304.854791
std,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,47.000000,76.000000,27.000000,59.000000,46.000000
max,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [15]:
train_df[spend_cols] = train_df[spend_cols].fillna(0)

In [16]:
train_df['TotalSpend'] = train_df[spend_cols].sum(axis=1)

In [17]:
train_df['TotalSpend'].describe()

count     8693.000000
mean      1440.866329
std       2803.045694
min          0.000000
25%          0.000000
50%        716.000000
75%       1441.000000
max      35987.000000
Name: TotalSpend, dtype: float64

In [18]:
train_df['IsSpender'] = (train_df['TotalSpend'] > 0).astype(int)

In [19]:
train_df['IsSpender'].value_counts()

IsSpender
1    5040
0    3653
Name: count, dtype: int64

In [20]:
train_df['Cabin'].head(10)

0    B/0/P
1    F/0/S
2    A/0/S
3    A/0/S
4    F/1/S
5    F/0/P
6    F/2/S
7    G/0/S
8    F/3/S
9    B/1/P
Name: Cabin, dtype: object

In [21]:
train_df[['Deck','CabinNum','Side']] = train_df['Cabin'].str.split('/', expand=True)

In [22]:
train_df[['Deck','Side']].value_counts().head()

Deck  Side
F     P       1438
      S       1356
G     S       1283
      P       1276
E     S        447
Name: count, dtype: int64

In [23]:
pd.crosstab(train_df['Deck'], train_df['Transported'], normalize='index')

Transported,False,True
Deck,,
A,0.503906,0.496094
B,0.265725,0.734275
C,0.319946,0.680054
D,0.566946,0.433054
E,0.642694,0.357306
F,0.560129,0.439871
G,0.483783,0.516217
T,0.800000,0.200000


In [24]:
pd.crosstab(train_df['Side'], train_df['Transported'], normalize='index')

Transported,False,True
Side,,
P,0.548740,0.451260
S,0.444963,0.555037


In [25]:
train_df.drop(columns=['Cabin', 'CabinNum', 'Name', 'PassengerId'], inplace=True)

In [26]:
train_df.columns

Index(['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService',
       'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported',
       'TotalSpend', 'IsSpender', 'Deck', 'Side'],
      dtype='object')

In [27]:
train_df.isnull().sum()

HomePlanet      201
CryoSleep       217
Destination     182
Age             179
VIP             203
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Transported       0
TotalSpend        0
IsSpender         0
Deck            199
Side            199
dtype: int64

In [28]:
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())

In [29]:
cat_cols = ['HomePlanet','CryoSleep','Destination','VIP','Deck','Side']

for col in cat_cols:
    train_df[col] = train_df[col].fillna('Unknown')

In [30]:
train_df.isnull().sum()

HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
TotalSpend      0
IsSpender       0
Deck            0
Side            0
dtype: int64

### One-hot encoding

In [31]:
train_encoded = pd.get_dummies(
    train_df,
    columns=['HomePlanet','CryoSleep','Destination','VIP','Deck','Side'],
    drop_first=True
)

In [32]:
train_encoded.shape

(8693, 29)

In [33]:
X = train_encoded.drop('Transported', axis=1)

In [34]:
y = train_encoded['Transported']

In [35]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [36]:
X_train.shape, X_val.shape

((6954, 28), (1739, 28))

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [38]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [39]:
y_pred = model.predict(X_val_scaled)

In [40]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

accuracy, f1

(0.7918343875790684, 0.7952488687782805)

In [41]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

test_df[spend_cols] = test_df[spend_cols].fillna(0)
test_df['TotalSpend'] = test_df[spend_cols].sum(axis=1)
test_df['IsSpender'] = (test_df['TotalSpend'] > 0).astype(int)

In [42]:
test_df[['Deck','CabinNum','Side']] = test_df['Cabin'].str.split('/', expand=True)

In [43]:
test_passenger_ids = test_df['PassengerId']  # save for submission

test_df.drop(
    columns=['PassengerId','Name','Cabin','CabinNum'],
    inplace=True
)

In [44]:
test_df['Age'] = test_df['Age'].fillna(train_df['Age'].median())

cat_cols = ['HomePlanet','CryoSleep','Destination','VIP','Deck','Side']
for col in cat_cols:
    test_df[col] = test_df[col].fillna('Unknown')

In [45]:
test_encoded = pd.get_dummies(
    test_df,
    columns=['HomePlanet','CryoSleep','Destination','VIP','Deck','Side'],
    drop_first=True
)

In [46]:
test_encoded = test_encoded.reindex(columns=X.columns, fill_value=0)

In [47]:
test_scaled = scaler.transform(test_encoded)

In [48]:
test_predictions = model.predict(test_scaled)

In [49]:
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': test_predictions
})

submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [50]:
submission.to_csv("submission.csv", index=False)

### RandomForest

In [51]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [52]:
rf_pred = rf_model.predict(X_val)

In [53]:
from sklearn.metrics import accuracy_score, f1_score

rf_accuracy = accuracy_score(y_val, rf_pred)
rf_f1 = f1_score(y_val, rf_pred)

rf_accuracy, rf_f1

(0.8021851638872916, 0.7976470588235294)